# ShoppingBench 全量离线向量索引（Kaggle · 双 T4）

在右侧 **Settings** 里把 **Accelerator** 设为 **GPU T4 x2**，**Internet** 设为 **On**。

建议把下面内容打包成一个**私有 Kaggle 数据集**，用 **Add Data** 挂进来，这样每个会话都不用重新下载：

| 文件 | 大小 | 是否必需 |
| --- | --- | --- |
| `kaggle_bundle.zip`（本地 `make_kaggle_bundle.py` 生成） | 很小 | 必需 |
| `documents.part1.jsonl.gz` + `documents.part2.jsonl.gz` | 约 1.5 GB | 可选，缺失则联网下载官方单文件 |
| `bge-m3/`（模型目录） | 约 2.3 GB | 可选，缺失则联网下载 |

Kaggle 没有 Google Drive，检查点写在 `/kaggle/working`。跑完后点 **Save Version**，产物会出现在 notebook 的 **Output** 标签供下载。


In [ ]:
import subprocess, sys
from pathlib import Path

INPUT = Path('/kaggle/input')


def find_code_root():
    # Kaggle 上传 ZIP 会自动解压。挂载点深度不固定：可能是 /kaggle/input/<slug>/，
    # 也可能是 /kaggle/input/datasets/<owner>/<slug>/，所以必须用 rglob 递归查找。
    hits = sorted(
        (p for p in INPUT.rglob('Shopping_agent')
         if p.is_dir() and (p / 'offline_embedding' / 'build_gpu.py').is_file()),
        key=lambda p: len(p.parts),
    )
    assert hits, (
        f'没有在 {INPUT} 下找到含 offline_embedding/build_gpu.py 的 Shopping_agent 目录，'
        '请确认含 kaggle_bundle.zip 的私有数据集已通过 Add Data 挂上'
    )
    return hits[0]


ROOT = find_code_root()
print('代码根目录:', ROOT)

# 盘点数据和模型是否就位——缺失的会在下一个单元联网下载。
# 口径与 download.py 保持一致：bge-m3 目录必须同时有 modules.json 和 config.json。
shards = sorted({p.name for p in INPUT.rglob('documents.jsonl.gz')}
                | {p.name for p in INPUT.rglob('documents.part*.jsonl.gz')})
models = sorted(
    str(d.relative_to(INPUT)) for d in INPUT.rglob('bge-m3')
    if d.is_dir() and (d / 'modules.json').is_file() and (d / 'config.json').is_file()
)
print('原始数据分片:', shards or '（缺失，将联网下载官方单文件 ~1.5GB）')
print('已挂载模型:', models or '（缺失，将联网下载 bge-m3 ~2.3GB）')

subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q',
     'sentence-transformers>=3,<6', 'faiss-cpu>=1.8,<2', 'huggingface_hub>=0.25,<1'],
    check=True,
)
print('依赖安装完成')


In [ ]:
import runpy, sys, torch
from pathlib import Path

# 复用上一个单元定位到的 ROOT；若单独运行本单元则重新查找，避免写死 /kaggle/working。
try:
    ROOT
except NameError:
    INPUT = Path('/kaggle/input')
    ROOT = sorted(
        (p for p in INPUT.rglob('Shopping_agent')
         if p.is_dir() and (p / 'offline_embedding' / 'build_gpu.py').is_file()),
        key=lambda p: len(p.parts),
    )[0]
print('代码根目录:', ROOT)

WORK = Path('/kaggle/working/shopping_work')     # 数据与模型，随 Save Version 保存
CKPT = Path('/kaggle/working/checkpoints')       # 断点，会话中断后可从这里续跑
for path in (WORK, CKPT):
    path.mkdir(parents=True, exist_ok=True)

assert torch.cuda.is_available(), '未检测到 GPU；请在 Settings 里把 Accelerator 设为 GPU T4 x2'
print('GPU 数量:', torch.cuda.device_count())
for index in range(torch.cuda.device_count()):
    props = torch.cuda.get_device_properties(index)
    print(f'  cuda:{index} {torch.cuda.get_device_name(index)} {props.total_memory / 1e9:.1f} GB')


def run_script(name, *arguments):
    script = ROOT / 'offline_embedding' / name
    if not script.is_file():
        raise FileNotFoundError(f'找不到脚本 {script}；请确认 ROOT 指向含代码的 Shopping_agent 目录')
    previous = sys.argv
    try:
        sys.argv = [name, *map(str, arguments)]
        runpy.run_path(str(script), run_name='__main__')
    finally:
        sys.argv = previous


In [ ]:
# 下载/复用原始数据与 BGE-M3，再规范化成 products.jsonl。
# 这一步是单线程解析 gzip，约需十几分钟；已存在时会自动跳过。
run_script('download.py', '--work-dir', WORK)
if not (WORK / 'export' / 'products.jsonl').is_file():
    run_script('prepare_full.py', '--work-dir', WORK)
else:
    print('已有 products.jsonl，跳过规范化')


In [ ]:
# 先用 2000 条实测单卡与双卡的吞吐，再决定用哪组参数跑全量（约 2 分钟）。
run_script('bench_gpu.py', '--work-dir', WORK, '--sample', 2000)


In [ ]:
# 全量构建。默认 --parallel auto：检测到两块 GPU 就用每卡一份模型 + 多线程。
# 断点在 /kaggle/working/checkpoints，会话断开后重跑本单元会接着算。
run_script(
    'build_gpu.py',
    '--work-dir', WORK,
    '--checkpoint-dir', CKPT,
    '--parallel', 'auto',
    '--precision', 'fp16',
    '--attn', 'sdpa',
    '--batch-size', 32,
)


In [ ]:
from pathlib import Path

WORK = Path('/kaggle/working/shopping_work')
index_dir = WORK / 'export' / 'faiss_bge_m3'
products = WORK / 'export' / 'products.jsonl'

if index_dir.is_dir():
    print('索引目录:', index_dir)
    for path in sorted(index_dir.iterdir()):
        print(f'  {path.name}  {path.stat().st_size / 1e6:.1f} MB')
else:
    print('索引目录还不存在:', index_dir, '（构建单元尚未跑完）')

print('商品文件:', products, '存在' if products.is_file() else '（缺失）')
print()
print('点 Save Version 后到 Output 标签下载 faiss_bge_m3/ 与 products.jsonl，')
print('把 export 内的内容复制到本地 Shopping_agent/data/（不要把 export 文件夹本身套进去）。')
print('若本地已用同一份原始分片跑过 prepare_full.py，可以只下载 faiss_bge_m3/。')


In [ ]:
# 可选但推荐：删掉不需要带回本地的中间产物，Save Version 会快很多。
# raw 原始分片、bge-m3 模型（本地已有 HF 缓存）、checkpoints（只是断点）都不需要下载。
import shutil
from pathlib import Path

WORK = Path('/kaggle/working/shopping_work')
shutil.rmtree(WORK / 'raw', ignore_errors=True)
shutil.rmtree(WORK / 'export' / 'models', ignore_errors=True)
shutil.rmtree('/kaggle/working/checkpoints', ignore_errors=True)

print('清理后 /kaggle/working 剩余：')
total = 0
for path in sorted(Path('/kaggle/working').rglob('*')):
    if path.is_file():
        size = path.stat().st_size
        total += size
        print(f'  {size/1e6:8.1f} MB  {path.relative_to("/kaggle/working")}')
print(f'  {"-"*12}')
print(f'  {total/1e6:8.1f} MB  合计')
print()
print('现在点 Save Version，到 Output 标签下载 faiss_bge_m3/ 与 products.jsonl。')
